[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyTorch and TensorFlow, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)

# Gradient Descent by Hand &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import contextlib
import os
import warnings

import numpy as np
import torch


@contextlib.contextmanager
def c_messages_hidden():
    """Send what C and C++ code writes to the error stream to os.devnull while the block runs."""
    saved = os.dup(2)
    devnull = os.open(os.devnull, os.O_WRONLY)
    os.dup2(devnull, 2)
    try:
        yield
    finally:
        os.dup2(saved, 2)
        os.close(devnull)
        os.close(saved)


os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
with c_messages_hidden():
    import tensorflow as tf
    import keras


def station_network_arrays(n=1200, seed=2025):
    """The made-up stations of the scikit-learn guide, as arrays: four features and two targets."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    shift = np.where(coast_km < 3, 0.8,
            np.where(elevation > 700, -0.5,
            np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), 0.3, 0.0)))
    rng.choice(4, n, p=[0.4, 0.3, 0.1, 0.2])   # each station's owner, which this guide leaves out
    rng.integers(1950, 2025, n)                 # and the year it was installed
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    features = np.column_stack([latitude.round(2), longitude.round(2), elevation.round(0), coast_km.round(1)])
    return features.astype("float32"), jan_mean.round(1).astype("float32"), (may_mean < 2.5).astype("float32")


FEATURES = ["latitude", "longitude", "elevation_m", "coast_km"]
features, jan_mean, snow_on_may_1 = station_network_arrays()
order = np.random.default_rng(0).permutation(len(features))
train_rows, valid_rows, test_rows = order[:800], order[800:1000], order[1000:]
mean, std = features[train_rows].mean(axis=0), features[train_rows].std(axis=0)
scaled = (features - mean) / std

print("torch", torch.__version__.split("+")[0], "| tensorflow", tf.__version__, "|", len(features), "stations")

torch 2.11.0 | tensorflow 2.21.0 | 1200 stations


The worked examples' tensors and `train`, rebuilt here so that each answer runs on its own.

In [2]:
X_train = torch.from_numpy(scaled[train_rows])
y_train = torch.from_numpy(jan_mean[train_rows])
X_raw = torch.from_numpy(features[train_rows])


def train(lr, steps, X=X_train):
    """Fit the four-feature linear model by gradient descent and return the loss before each step."""
    w = torch.zeros(4, requires_grad=True)
    b = torch.tensor(0.0, requires_grad=True)
    losses = []
    for step in range(steps):
        loss = ((X @ w + b - y_train) ** 2).mean()
        losses.append(loss.item())
        loss.backward()
        with torch.no_grad():
            w -= lr * w.grad
            b -= lr * b.grad
        w.grad.zero_()
        b.grad.zero_()
    return losses

**1.** Run `train` with a learning rate of 0.3 for 201 steps, and print the loss at steps 10, 50 and 200 beside the rows for 0.1 and 0.5 in the worked examples.

In [3]:
print("rate    step 10  step 50  step 200")
for lr in (0.1, 0.3, 0.5):
    losses = train(lr, 201)
    print(f"{lr:<7} {losses[10]:<8.3f} {losses[50]:<8.3f} {losses[200]:.3f}")

rate    step 10  step 50  step 200
0.1     1.347    1.019    0.996
0.3     1.027    1.000    0.992
0.5     2.962    1.159    0.992


A rate of 0.3 has the lowest loss of the three at step 10, 1.027, and reaches the exact answer's 0.992 by step 200, as 0.5 does.

**2.** Print the loss after 200 steps for learning rates of 0.50, 0.505, 0.51 and 0.52, formatted with `:.3g`, and find where the loop stops converging.

In [4]:
for lr in (0.50, 0.505, 0.51, 0.52):
    print(lr, f"{train(lr, 201)[200]:.3g}")

0.5 0.992
0.505 1.04
0.51 131
0.52 5.86e+08


The loop stops converging somewhere between 0.5 and 0.51. At 0.505 the loss is 1.04 after 200 steps, above 0.5's 0.992, at 0.51 it is 131, and at 0.52 it is 5.86e+08.

**3.** Run the mini-batch loop with batches of 100 stations in place of 32, from `torch.Generator().manual_seed(0)`, and print the loss on all 800 stations after epochs 5, 10 and 20. How many steps is an epoch now?

In [5]:
generator = torch.Generator().manual_seed(0)
w = torch.zeros(4, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
steps = 0
for epoch in range(1, 21):
    shuffled = torch.randperm(len(train_rows), generator=generator)
    for start in range(0, len(train_rows), 100):
        batch = shuffled[start:start + 100]
        loss = ((X_train[batch] @ w + b - y_train[batch]) ** 2).mean()
        loss.backward()
        with torch.no_grad():
            w -= 0.1 * w.grad
            b -= 0.1 * b.grad
        w.grad.zero_()
        b.grad.zero_()
        steps += 1
    if epoch in (5, 10, 20):
        with torch.no_grad():
            print(f"epoch {epoch:<3} all 800 {((X_train @ w + b - y_train) ** 2).mean().item():.3f}")
print("steps per epoch:", steps // 20)

epoch 5   all 800 1.024
epoch 10  all 800 1.012
epoch 20  all 800 1.000
steps per epoch: 8


An epoch is now 8 steps. After 20 epochs, 160 steps, the loss on all 800 stations is 1.000, where batches of 32 reached 0.999 in 500 steps.

**4.** Train on the features in their own units, `X_raw`, with a learning rate of 1e-6 for 2,001 steps, and print the loss at steps 200 and 2,000 beside the standardized loop's.

In [6]:
own_units = train(1e-6, 2001, X_raw)
standardized = train(0.1, 2001)
print("own units, 1e-6:   ", round(own_units[200], 3), round(own_units[2000], 3))
print("standardized, 0.1: ", round(standardized[200], 3), round(standardized[2000], 3))

own units, 1e-6:    5.704 3.926
standardized, 0.1:  0.996 0.992


In their own units, the loss is 5.704 at step 200 and 3.926 at step 2,000, where the standardized loop had reached 0.996 and 0.992.

**5.** Write the TensorFlow loop with mini-batches of 32, shuffling the row numbers each epoch with `permutation` from one `np.random.default_rng(0)` made before the loop and taking each batch with `tf.gather`, and print the loss on all 800 stations after 20 epochs.

In [7]:
X_train_tf = tf.constant(scaled[train_rows])
y_train_tf = tf.constant(jan_mean[train_rows])
rng = np.random.default_rng(0)

w = tf.Variable(tf.zeros(4))
b = tf.Variable(0.0)
for epoch in range(20):
    shuffled = rng.permutation(len(train_rows))
    for start in range(0, len(train_rows), 32):
        batch = shuffled[start:start + 32]
        X_batch = tf.gather(X_train_tf, batch)
        y_batch = tf.gather(y_train_tf, batch)
        with tf.GradientTape() as tape:
            loss = tf.reduce_mean((tf.linalg.matvec(X_batch, w) + b - y_batch) ** 2)
        dw, db = tape.gradient(loss, [w, b])
        w.assign_sub(0.1 * dw)
        b.assign_sub(0.1 * db)

all_loss = tf.reduce_mean((tf.linalg.matvec(X_train_tf, w) + b - y_train_tf) ** 2)
print("loss on all 800 after 20 epochs:", round(float(all_loss), 3))

loss on all 800 after 20 epochs: 1.006


The loss on all 800 stations is 1.006 after 20 epochs. The batches come from NumPy's generator here, in other orders than PyTorch's, so the loss differs from the PyTorch loop's 0.999.

---

&#8592; **Back to:** [Gradient Descent by Hand](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/05-gradient-descent-by-hand.ipynb)  &nbsp;&middot;&nbsp;  [PyTorch and TensorFlow, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)
